# Notebook 05: The Logit Lens & Tuned Lens

Want to see what a transformer is "thinking" at each layer? The **logit lens** and **tuned lens** let you peek inside the residual stream at every layer to watch the model's predictions evolve in real time. These are some of the simplest and most useful interpretability tools you can reach for.

**Prerequisites**: Notebook 01 (transformer circuits -- residual stream, attention, MLPs).

**What we'll cover**:
1. The logit lens: projecting intermediate residual streams through the unembedding
2. Top-k predictions at each layer
3. The tuned lens: fixing the logit lens with learned affine transforms
4. What the lens reveals about model computation
5. Multi-prompt comparison

## Section 1: The Logit Lens

The **logit lens** ([nostalgebraist, 2020](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens)) is dead simple but surprisingly powerful: at each layer, project the residual stream through the model's final unembedding matrix to see what tokens the model would predict at that intermediate stage.

**Intuition:** The residual stream accumulates information layer by layer. By "peeking" at intermediate layers through the unembedding, you can see the model's evolving "beliefs" about what comes next.

**Mathematically:** For residual stream $h_l$ at layer $l$, we compute $\text{logits}_l = h_l \cdot W_U$ (where $W_U$ is the unembedding matrix), then look at the top predicted tokens.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained("gpt2-small")
model.eval()

prompt = "The Eiffel Tower is located in the city of"
logits, cache = model.run_with_cache(prompt)
tokens = [model.tokenizer.decode(t) for t in model.to_tokens(prompt)[0]]

# Apply logit lens at each layer
n_layers = model.cfg.n_layers
logit_lens_predictions = []
logit_lens_logits = []

for layer in range(n_layers + 1):  # +1 for after final layer norm
    if layer < n_layers:
        resid = cache[f"blocks.{layer}.hook_resid_post"][0]  # (seq, d_model)
    else:
        # After final layer norm (this should match the actual output)
        resid = cache["ln_final.hook_normalized"][0]
    
    # Apply unembedding
    layer_logits = resid @ model.W_U + model.b_U  # (seq, vocab)
    logit_lens_logits.append(layer_logits.detach().cpu())
    
    # Get top prediction at the last position
    top_token = layer_logits[-1].argmax().item()
    top_prob = torch.softmax(layer_logits[-1], dim=0).max().item()
    logit_lens_predictions.append((model.to_string([top_token]), top_prob))

# Print evolution of predictions at last position
print(f"Prompt: '{prompt}'\n")
print("Layer-by-layer predictions for the NEXT token:")
print("-" * 50)
for layer, (token, prob) in enumerate(logit_lens_predictions):
    marker = " ←" if token.strip() == "Paris" else ""
    print(f"Layer {layer:2d}: '{token}' (prob: {prob:.3f}){marker}")

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# For each layer and position, compute probability of the final answer
final_predictions = logits[0].argmax(dim=-1)  # What the model actually predicts

# Build matrix: P(correct final token | layer l, position p)
correct_token_probs = torch.zeros(n_layers + 1, len(tokens))
# Also store the target token labels for hover info
target_token_labels = []

for pos in range(len(tokens)):
    if pos < len(tokens) - 1:
        target = model.to_tokens(prompt)[0, pos + 1].item()
    else:
        target = final_predictions[pos].item()
    target_token_labels.append(model.tokenizer.decode(target))

for layer in range(n_layers + 1):
    probs = torch.softmax(logit_lens_logits[layer], dim=-1)
    for pos in range(len(tokens)):
        if pos < len(tokens) - 1:
            target = model.to_tokens(prompt)[0, pos + 1].item()
        else:
            target = final_predictions[pos].item()
        correct_token_probs[layer, pos] = probs[pos, target].item()

# Build custom hover text with (layer, position token, target token, probability)
hover_text = []
for layer in range(n_layers + 1):
    row = []
    for pos in range(len(tokens)):
        row.append(
            f"Layer: {layer}<br>Position: {tokens[pos]}<br>"
            f"Target: {target_token_labels[pos]}<br>"
            f"P(target): {correct_token_probs[layer, pos]:.4f}"
        )
    hover_text.append(row)

fig = go.Figure(data=go.Heatmap(
    z=correct_token_probs.numpy(),
    x=tokens,
    y=list(range(n_layers + 1)),
    colorscale="Viridis",
    zmin=0,
    zmax=1,
    colorbar=dict(title="Probability"),
    hovertext=hover_text,
    hoverinfo="text",
))
fig.update_layout(
    title="Logit Lens: P(correct next token) at each layer",
    xaxis_title="Token Position",
    yaxis_title="Layer",
    height=600,
    width=900,
)
fig.show()

## Section 2: Top-k Predictions at Each Layer

Looking at just the top prediction isn't enough. Let's examine the top-k at each layer to see how the model's "beliefs" evolve. Sometimes the correct answer sits in the top-5 long before it becomes the top-1.

In [ ]:
# Show top-5 predictions at each layer for the last position
print(f"Top-5 predictions for next token after: '{prompt}'")
print("=" * 70)

for layer in range(0, n_layers + 1, 2):  # Every other layer for brevity
    probs = torch.softmax(logit_lens_logits[layer][-1], dim=0)
    top5 = probs.topk(5)
    predictions = [(model.to_string([idx.item()]), val.item()) 
                   for val, idx in zip(top5.values, top5.indices)]
    pred_str = " | ".join([f"'{t}'({p:.2f})" for t, p in predictions])
    print(f"L{layer:2d}: {pred_str}")

## Section 3: The Tuned Lens

Fair warning: the logit lens is an approximation. Early layers produce representations in a different "coordinate system" than the final layer. The unembedding matrix was trained to interpret the *final* residual stream, not intermediate ones.

The **tuned lens** ([Belrose et al., 2023](https://arxiv.org/abs/2303.08112)) fixes this by learning a simple affine transformation for each layer:

$$\text{logits}_l = \text{LayerNorm}(A_l \cdot h_l + b_l) \; W_U$$

**Note:** The final layer norm is part of the tuned lens computation. The affine transform maps the intermediate residual stream to the final residual stream space, after which the model's final layer norm is applied before the unembedding projection.

Where $A_l$ and $b_l$ are learned per-layer parameters (trained on a held-out dataset to predict the model's actual output). This is strictly more expressive and consistently outperforms the logit lens.

**Key findings from the tuned lens paper:**
- Predictions converge gradually, not in sudden jumps
- Different layers specialize in different aspects of prediction (e.g., syntactic vs. semantic processing)
- The lens reveals a consistent "iterative inference" process

In [ ]:
import plotly.graph_objects as go

# The full tuned lens requires training affine probes on a dataset.
# Here we demonstrate the concept with a lightweight version.
# For production use, install: pip install tuned-lens

# Simplified: Learn a per-layer scaling factor (diagonal affine)
# This is weaker than the full tuned lens but illustrates the idea.

# First, let's compare logit lens accuracy vs layer
logit_lens_accuracy = []
for layer in range(n_layers + 1):
    probs = torch.softmax(logit_lens_logits[layer], dim=-1)
    # Check if top prediction matches final model output at each position
    predicted = logit_lens_logits[layer].argmax(dim=-1)
    final = logits[0].argmax(dim=-1).cpu()
    accuracy = (predicted == final).float().mean().item()
    logit_lens_accuracy.append(accuracy)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(n_layers + 1)),
    y=logit_lens_accuracy,
    mode="lines+markers",
    marker=dict(size=7, color="royalblue"),
    line=dict(color="royalblue"),
    name="Logit Lens",
    hovertemplate="Layer: %{x}<br>Agreement: %{y:.3f}<extra></extra>",
))
fig.update_layout(
    title="Logit Lens: Agreement with Model Output by Layer",
    xaxis_title="Layer",
    yaxis_title="Agreement with Final Output",
    height=450,
    width=750,
    showlegend=True,
)
fig.show()

print("Note: A full tuned lens (with learned affine transforms) would show")
print("higher agreement at early layers. Install `tuned-lens` for the full version.")

## Section 4: What the Lens Reveals

The logit/tuned lens shows you several important things:

1. **Iterative refinement**: The model doesn't "know" the answer immediately. Predictions evolve layer by layer, with each layer making incremental adjustments.

2. **Different layers, different roles**: Early layers tend to predict syntactically plausible next tokens, while later layers incorporate world knowledge and context. (This is a general finding from multiple probing studies, not specific to the tuned lens.)

3. **Sudden jumps**: For some tokens, predictions change abruptly at a specific layer -- that tells you a key computation happens there (great for targeted mechanistic analysis).

4. **Layer redundancy**: Some layers barely change the prediction, suggesting they're doing other work (or are redundant).

In [ ]:
import plotly.graph_objects as go

# Measure prediction change between consecutive layers (KL divergence)
kl_between_layers = []

for layer in range(1, n_layers + 1):
    prev_probs = torch.softmax(logit_lens_logits[layer - 1][-1], dim=0)
    curr_probs = torch.softmax(logit_lens_logits[layer][-1], dim=0)
    
    # KL(curr || prev) — how much the prediction changed
    kl = (curr_probs * (curr_probs.log() - prev_probs.log())).sum().item()
    kl_between_layers.append(kl)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=list(range(1, n_layers + 1)),
    y=kl_between_layers,
    marker_color="steelblue",
    hovertemplate="Layer: %{x}<br>KL Divergence: %{y:.4f}<extra></extra>",
))
fig.update_layout(
    title="Prediction Change Between Layers (Last Token)",
    xaxis_title="Layer",
    yaxis_title="KL Divergence from Previous Layer",
    height=450,
    width=750,
)
fig.show()

# Identify layers with biggest jumps
top_change_layers = np.argsort(kl_between_layers)[-3:][::-1]
print("Layers with biggest prediction changes:")
for l in top_change_layers:
    prev_pred = model.to_string([logit_lens_logits[l][-1].argmax().item()])
    curr_pred = model.to_string([logit_lens_logits[l+1][-1].argmax().item()])
    print(f"  Layer {l+1}: '{prev_pred}' -> '{curr_pred}' (KL={kl_between_layers[l]:.4f})")

## Section 5: Multi-Prompt Comparison

Let's compare the logit lens across different types of prompts to see how prediction dynamics differ. Factual recall, syntactic completion, arithmetic, and name completion may each resolve at different depths in the network.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

prompts = [
    ("Factual", "The capital of Germany is"),
    ("Syntactic", "The cat sat on the"),
    ("Reasoning", "If 2 + 2 = 4, then 3 + 3 ="),
    ("Name completion", "The first president of the US was George"),
]

fig = make_subplots(
    rows=1, cols=len(prompts),
    subplot_titles=[f"{label}" for label, _ in prompts],
    horizontal_spacing=0.06,
)

for idx, (label, prompt) in enumerate(prompts):
    logits_p, cache_p = model.run_with_cache(prompt)
    final_pred = logits_p[0, -1].argmax().item()
    final_token = model.to_string([final_pred])
    
    # Probability of final answer at each layer
    probs_by_layer = []
    for layer in range(n_layers + 1):
        if layer < n_layers:
            resid = cache_p[f"blocks.{layer}.hook_resid_post"][0, -1]
        else:
            resid = cache_p["ln_final.hook_normalized"][0, -1]
        layer_logits_p = resid @ model.W_U + model.b_U
        prob = torch.softmax(layer_logits_p, dim=0)[final_pred].item()
        probs_by_layer.append(prob)
    
    fig.add_trace(
        go.Scatter(
            x=list(range(n_layers + 1)),
            y=probs_by_layer,
            mode="lines+markers",
            marker=dict(size=4),
            name=f"{label} -> '{final_token}'",
            hovertemplate=f"Layer: %{{x}}<br>P('{final_token}'): %{{y:.4f}}<extra>{label}</extra>",
        ),
        row=1, col=idx + 1,
    )
    fig.update_yaxes(range=[0, 1], title_text="P(final answer)" if idx == 0 else "", row=1, col=idx + 1)
    fig.update_xaxes(title_text="Layer", row=1, col=idx + 1)

fig.update_layout(
    title_text="Logit Lens: When does the model 'know' the answer?",
    height=400,
    width=1100,
    showlegend=True,
)
fig.show()

---
### Running Example: IOI — When Does the Model Know?

This is part of our **running example** investigating how GPT-2-small handles the Indirect Object Identification (IOI) task across all techniques in this guide.

**The task**: "When Mary and John went to the store, John gave a drink to" — the model should predict "Mary".

Here we apply the logit lens to the IOI prompt to see at which layer the model first starts predicting "Mary". This reveals the depth at which the IOI circuit's name mover heads have done their work.

In [ ]:
# Running Example: IOI — At which layer does the model start predicting "Mary"?
prompt = "When Mary and John went to the store, John gave a drink to"
logits, cache = model.run_with_cache(prompt)
mary_token = model.to_single_token(" Mary")

print(f"Prompt: '{prompt}'")
print(f"Target token: ' Mary' (id={mary_token})")
print(f"\nLogit lens — P(Mary) and rank at each layer:")
print("-" * 55)

for layer in range(model.cfg.n_layers + 1):
    if layer < model.cfg.n_layers:
        resid = cache[f"blocks.{layer}.hook_resid_post"][0, -1]
    else:
        resid = cache["ln_final.hook_normalized"][0, -1]
    layer_logits = resid @ model.W_U + model.b_U
    mary_prob = torch.softmax(layer_logits, dim=0)[mary_token].item()
    mary_rank = (layer_logits > layer_logits[mary_token]).sum().item()
    marker = " <-- first notable" if mary_prob > 0.01 and (layer == 0 or mary_rank < 50) else ""
    print(f"Layer {layer:2d}: P(Mary) = {mary_prob:.4f}, rank = {mary_rank}{marker}")

## Exercises

### Exercise 1: Track a Specific Prediction

Pick a factual prompt like `"The Eiffel Tower is located in"` where the expected answer is `"Paris"`. Apply the logit lens at every layer. At which layer does `"Paris"` first enter the top-5 predictions? Plot the rank of `"Paris"` across layers.

<details>
<summary>Hint</summary>

For each layer, project the residual stream at the last token position through `W_U` (the unembedding matrix) and look at the argsort of the resulting logits. Use `cache['blocks.{layer}.hook_resid_post']` to get the residual stream at each layer. The rank of "Paris" is the number of tokens with a higher logit value.

</details>

In [ ]:
import plotly.graph_objects as go

prompt = "The Eiffel Tower is located in"
# Run with cache
logits, cache = model.run_with_cache(prompt)
paris_token = model.to_single_token(" Paris")

# For each layer:
#   Project residual stream at last position through W_U
#   Get top-5 tokens and rank of "Paris"
ranks = []
for layer in range(model.cfg.n_layers + 1):
    if layer < model.cfg.n_layers:
        resid = cache[f"blocks.{layer}.hook_resid_post"][0, -1]
    else:
        resid = cache["ln_final.hook_normalized"][0, -1]
    layer_logits = resid @ model.W_U + model.b_U
    # Rank of Paris = number of tokens with higher logit
    paris_rank = (layer_logits > layer_logits[paris_token]).sum().item()
    ranks.append(paris_rank)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(len(ranks))),
    y=ranks,
    mode="lines+markers",
    marker=dict(size=7, color="red"),
    line=dict(color="red"),
    hovertemplate="Layer: %{x}<br>Rank: %{y}<extra></extra>",
    name="Rank of 'Paris'",
))
fig.add_hline(y=5, line_dash="dash", line_color="blue", opacity=0.5, annotation_text="Top-5 threshold")
fig.update_layout(
    title="Logit Lens: Rank of 'Paris' across layers",
    xaxis_title="Layer",
    yaxis_title="Rank of 'Paris' (lower = better)",
    yaxis_autorange="reversed",
    height=450,
    width=750,
)
fig.show()

first_top5 = next((l for l, r in enumerate(ranks) if r < 5), None)
print(f"'Paris' first enters top-5 at layer {first_top5}")

### Exercise 2: Compare Logit Lens vs Tuned Lens

For the same prompt, compare raw logit lens predictions with a "DIY tuned lens" -- train a simple affine transform (a `nn.Linear` layer) per layer on 30 random prompts that maps the residual stream to better approximate the final output. Compare prediction accuracy between the raw logit lens and your tuned version.

<details>
<summary>Hint</summary>

For each layer, collect `(residual_post[layer], final_logits)` pairs across 30 prompts. Train a linear regression (or a single `nn.Linear` layer) to predict the final logits from the intermediate residual stream. This is a simplified version of the tuned lens. Compare how often the top-1 prediction matches the model's actual output at each layer.

</details>

In [ ]:
# Exercise 2: Compare Logit Lens vs DIY Tuned Lens
# We train a simple per-layer linear map from residual stream -> final residual stream

sample_prompts = [
    "The cat sat on the", "I went to the store and", "The weather today is",
    "She picked up the", "They decided to go to", "He said that the",
    "The quick brown fox", "In the beginning there", "We should probably not",
    "My favorite color is", "The largest country in", "Dogs are known for",
    "After the rain the", "She opened the door and", "The movie was really",
    "I think we should", "He ran across the", "The teacher asked the",
    "It was a dark and", "They flew to the", "The recipe calls for",
    "On Monday morning the", "She wrote a letter to", "The old man sat",
    "We drove through the", "The company announced a", "He picked up his",
    "The river flows through", "After dinner we went", "The president signed the",
]

# Collect residual streams and final-layer representations
print("Collecting activations from sample prompts...")
layer_resids = {l: [] for l in range(model.cfg.n_layers)}
final_resids = []

for p in sample_prompts:
    _, c = model.run_with_cache(p)
    for l in range(model.cfg.n_layers):
        layer_resids[l].append(c[f"blocks.{l}.hook_resid_post"][0, -1].detach())
    final_resids.append(c["ln_final.hook_normalized"][0, -1].detach())

final_resids = torch.stack(final_resids)  # (n_prompts, d_model)

# For each layer, train a linear map and compare accuracy with raw logit lens
# TODO: Try adding more prompts for a better tuned lens!
raw_accuracy = []
tuned_accuracy = []

for layer in range(model.cfg.n_layers):
    X = torch.stack(layer_resids[layer])  # (n_prompts, d_model)
    
    # Raw logit lens: project directly through W_U
    raw_logits = X @ model.W_U + model.b_U
    final_logits = final_resids @ model.W_U + model.b_U
    raw_match = (raw_logits.argmax(dim=-1) == final_logits.argmax(dim=-1)).float().mean().item()
    raw_accuracy.append(raw_match)
    
    # DIY tuned lens: learn a linear transform X -> final_resids, then apply W_U
    # Simple least-squares: W_tune = (X^T X)^{-1} X^T final_resids
    XtX = X.T @ X + 1e-4 * torch.eye(X.shape[1], device=X.device)  # regularize
    W_tune = torch.linalg.solve(XtX, X.T @ final_resids)
    tuned_pred = X @ W_tune
    tuned_logits = tuned_pred @ model.W_U + model.b_U
    tuned_match = (tuned_logits.argmax(dim=-1) == final_logits.argmax(dim=-1)).float().mean().item()
    tuned_accuracy.append(tuned_match)

plt.figure(figsize=(10, 5))
plt.plot(range(model.cfg.n_layers), raw_accuracy, 'bo-', label="Raw logit lens")
plt.plot(range(model.cfg.n_layers), tuned_accuracy, 'rs-', label="DIY tuned lens")
plt.xlabel("Layer")
plt.ylabel("Agreement with final output (top-1)")
plt.title("Raw Logit Lens vs. DIY Tuned Lens")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Note: With only ~30 prompts the tuned lens overfits. Use 1000+ for real results.")

## Section 6: Key Takeaways & Further Reading

**What you should remember:**
- The logit lens reveals the model's evolving predictions layer by layer
- The tuned lens improves accuracy by accounting for per-layer coordinate shifts
- Models perform iterative refinement -- predictions improve gradually
- Large prediction jumps at specific layers point to key computations (great targets for mech interp)
- Different types of knowledge (syntactic vs factual) resolve at different depths

**When to reach for the lens:**
- Identify which layers are important for specific behaviors
- Guide activation patching: focus on layers where predictions change
- Understand the "processing stages" of the model
- Debug model failures: see where the correct prediction goes wrong

**Further reading:**
- [nostalgebraist's blog post on the logit lens](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) (2020)
- [Eliciting Latent Predictions (Tuned Lens)](https://arxiv.org/abs/2303.08112) (Belrose et al., 2023)
- *(A reference previously listed here -- "LogitLens4LLMs", arXiv 2503.11667 -- could not be verified and has been removed.)*

**Next**: Notebook 06 -- Probing (what information is linearly decodable?)